# From HCD absorbers to a mock inference: a tutorial for `hcd_priya`

**PROVISIONAL (PI to refine).** This notebook has three parts.

1. **Scientific motivation.** The Lyman-alpha forest and the high-column-density (HCD) absorbers on the same
   sightlines: Lyman limit systems (LLS), sub-damped and damped Lyman-alpha absorbers (subDLAs, DLAs).
2. **The emulator.** What the published model `hcd_priya` 1.0.0 returns: the flux power spectrum (P1D) of four
   classes of sightline, the incidence of the absorbers, and a mixture of the classes.
3. **Mock inference.** A Gaussian likelihood on a mock made by the model itself, with four free parameters. It is
   a self-consistency demonstration: not an independent validation, not a result from data.

**To run it:** `pip install "hcd-priya[tutorials]"` in Python 3.11 or later, then open this file with Jupyter. The
trained model comes with the package. The few stored files the notebook reads are found in a clone of the
repository or downloaded once, each checked against its sha256 (next cell).

In [ ]:
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np

import hcd_analysis
import hcd_analysis.predictor as hp

print("hcd_analysis", hcd_analysis.__version__)

In [ ]:
# The stored files of this tutorial. Each is named by its path in the repository (a file of the GitHub release by
# its file name) and pinned by its sha256. Every URL is immutable: the tag v1.0.0, a file of the release v1.0.0,
# or one commit.
import hashlib
import os
import pathlib
import tempfile
import urllib.request

ASSETS = {
    "docs/figures/data/ho21/cddf_z225.txt": {
        "sha256": "feaffb7b88215e767ca53a41fb3f00b65c90aaf27745c28720b32976a9168a63",
        "url": "https://raw.githubusercontent.com/jibanCat/hcd_priya/v1.0.0/docs/figures/data/ho21/cddf_z225.txt",
    },
    "docs/figures/data/ho21/cddf_z253.txt": {
        "sha256": "62e93a1e1977b3d78b9f56475bd4390c1eee7eaf371986ca5ece2db338bb7cc7",
        "url": "https://raw.githubusercontent.com/jibanCat/hcd_priya/v1.0.0/docs/figures/data/ho21/cddf_z253.txt",
    },
    "docs/figures/data/ho21/cddf_z34.txt": {
        "sha256": "f21dca06cd5974a08de13632ef74109f5b78afc2921f86ec9b3c6cf9f838ad3a",
        "url": "https://raw.githubusercontent.com/jibanCat/hcd_priya/v1.0.0/docs/figures/data/ho21/cddf_z34.txt",
    },
    "docs/figures/data/ho21/dndx_all.txt": {
        "sha256": "fb4d86151b34240902dbd6e63914e7c98dbb367fb6880a0e6871c849345ca81c",
        "url": "https://raw.githubusercontent.com/jibanCat/hcd_priya/v1.0.0/docs/figures/data/ho21/dndx_all.txt",
    },
    "docs/figures/data/lls_incidence/lls_lz_literature.dat": {
        "sha256": "c7fd68c9effd2f0151fa5293ca57214a106c31e1e4611d63a6442b2b2ff47ae1",
        "url": "https://raw.githubusercontent.com/jibanCat/hcd_priya/v1.0.0/docs/figures/data/lls_incidence/lls_lz_literature.dat",
    },
    "docs/figures/data/lls_subdla/zafar_2013.txt": {
        "sha256": "f592ecfd74bb423eae15d9bf40c7aca3e3da61ee7bbc1dab9510e6edca4767bd",
        "url": "https://raw.githubusercontent.com/jibanCat/hcd_priya/v1.0.0/docs/figures/data/lls_subdla/zafar_2013.txt",
    },
    "docs/figures/data/lls_subdla/omeara2007_fn_all19p0.dat": {
        "sha256": "695fd5e611368249b17b8de0ccf42aab8ee5c74c68f0750f7997c6c3d81f70bc",
        "url": "https://raw.githubusercontent.com/jibanCat/hcd_priya/v1.0.0/docs/figures/data/lls_subdla/omeara2007_fn_all19p0.dat",
    },
    "docs/figures/data/lls_subdla/prochaska2010_lls_region.dat": {
        "sha256": "9afad095522ac3d25ea1b615df91b806758ab50b42268dd6fea37412d689b39b",
        "url": "https://raw.githubusercontent.com/jibanCat/hcd_priya/v1.0.0/docs/figures/data/lls_subdla/prochaska2010_lls_region.dat",
    },
    "docs/figures/data/class_ratio_ranges_z3p0_lf60.csv": {
        "sha256": "0f79f1e5260df432fbe3b69b32dd4d4ebca248e95f789b11c40be549ef926f05",
        "url": "https://raw.githubusercontent.com/jibanCat/hcd_priya/v1.0.0/docs/figures/data/class_ratio_ranges_z3p0_lf60.csv",
    },
    "hcd_priya_example_download.tar.gz": {
        "sha256": "27bd0f96c88ae195175351bbf05468188b8fdf745c99bac9e54bebf571e31d7a",
        "url": "https://github.com/jibanCat/hcd_priya/releases/download/v1.0.0/hcd_priya_example_download.tar.gz",
    },
}

# Inside a clone of the repository the files are on disk already. A local file is used when, and only when, its
# sha256 is the pinned one, so the notebook reads the same bytes inside a clone and outside one. This notebook's
# folder is two levels below the top of a clone.
LOCAL_ROOTS = (pathlib.Path.cwd(), pathlib.Path.cwd().parent.parent)


def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def tutorial_file(relpath):
    # The path of a stored file of the tutorial, with the pinned sha256: a local copy in a clone, a copy
    # downloaded earlier, or a new download into the cache folder (the environment variable
    # HCD_PRIYA_TUTORIAL_CACHE, default ~/.cache/hcd_priya_tutorial).
    if relpath not in ASSETS:
        raise RuntimeError(f"{relpath} is not a stored file of this tutorial (ASSETS names {len(ASSETS)} files)")
    pinned, url = ASSETS[relpath]["sha256"], ASSETS[relpath]["url"]
    for root in LOCAL_ROOTS:
        local = pathlib.Path(root) / relpath
        if local.is_file() and sha256_of(local) == pinned:
            return local
    cache = pathlib.Path(os.environ.get("HCD_PRIYA_TUTORIAL_CACHE", "~/.cache/hcd_priya_tutorial")).expanduser()
    target = cache / relpath
    if target.is_file() and sha256_of(target) == pinned:
        return target
    target.parent.mkdir(parents=True, exist_ok=True)
    # written under a temporary name and renamed only after the digest is checked: no partial file is left
    handle, partial = tempfile.mkstemp(dir=target.parent, prefix=target.name + ".", suffix=".part")
    digest = hashlib.sha256()
    try:
        try:
            with os.fdopen(handle, "wb") as out, urllib.request.urlopen(url, timeout=120) as response:
                for block in iter(lambda: response.read(1 << 20), b""):
                    out.write(block)
                    digest.update(block)
        except Exception as exc:
            raise RuntimeError(
                f"could not download the tutorial file {relpath} from {url} into the cache folder {cache}: {exc}"
            ) from exc
        if digest.hexdigest() != pinned:
            raise RuntimeError(
                f"the tutorial file {relpath} downloaded from {url} has sha256 {digest.hexdigest()}, "
                f"not the pinned {pinned}: it is refused, and nothing is kept in {cache}"
            )
        os.replace(partial, target)
    finally:
        if os.path.exists(partial):
            os.remove(partial)
    return target


print(len(ASSETS), "stored files are pinned by sha256")